# Week 1 Day 2, remixed - Loan approval at BrightPath Lending

This is a copy of the Week 1 Day 2 lab adapted from insurance claims to **consumer lending**.

An underwriter at BrightPath Lending has already decided each of the 41 loan applications in the queue: 6 routine, 19 hard and 16 harder. The harder ones sit on the boundaries of the rules, need two rules at once, or hide the income in the arithmetic. We send every application to a few models, along with the lending policy. Each model replies with a decision of approve, decline or refer, a rationale and an approved amount. Then we compare the answers with the underwriters' decisions and look at what each model cost.

At the end we give the same job to a LangChain agent that uses tools to pull the credit file and record its decision.

The data is in the `data` folder next to this notebook. `make_data.py` generates it and works out the correct decision for each application from the rules in `data/lending_policy.md`, so you can add your own cases or change the policy and regenerate.

## 4 parts:

PART 1: Examine the data

PART 2: Run with some models

PART 3: Optional: a larger model and Jev

PART 4: An Agent via LangChain

In [1]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from openrouter import OpenRouter
from pydantic import BaseModel, Field

## Connect to OpenRouter

The key comes from the `.env` file in the `undercover` folder. If a new account hits a rate limit, add `time.sleep(3)` after each LLM call.

In [2]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

OpenRouter API key found


## Part 1: Load and examine the data

`get_sample_data` returns the applications and the underwriters' decisions on them, keyed by application id. The model is sent each application and the lending policy.

Set `TINY = True` to keep 5 applications if you are using free models with a daily limit.

In [3]:
DATA = Path("data")
TINY_IDS = {"APP-5001", "APP-5007", "APP-5009", "APP-5027", "APP-5038"}


def load(name):
    """Read one of the JSON data files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False):
    """Return a list of applications, and the underwriters' decisions on them keyed by application id."""
    applications = load("applications.json")
    if tiny:
        applications = [application for application in applications if application["application_id"] in TINY_IDS]
    chosen = {application["application_id"] for application in applications}
    labels = {decision["application_id"]: decision for decision in load("decisions.json") if decision["application_id"] in chosen}
    return applications, labels

In [4]:
TINY = False

applications, labels = get_sample_data(tiny=TINY)
lending_policy = (DATA / "lending_policy.md").read_text(encoding="utf-8")

print(f"{len(applications)} applications, {len(labels)} labels")
print(f"The lending policy is {len(lending_policy.split()):,} words")

41 applications, 41 labels
The lending policy is 733 words


In [5]:
applications[0]

{'application_id': 'APP-5001',
 'applied_on': '2026-10-01',
 'product': 'personal',
 'amount_requested': 10000,
 'term_months': 36,
 'purpose': 'home improvements',
 'story': 'I want to replace the windows in my house before winter.',
 'applicant': {'name': 'Maya Patel',
  'age': 34,
  'residency': 'citizen',
  'employment_type': 'employee',
  'employer': 'Northwind Logistics',
  'months_in_current_job_or_business': 52,
  'stated_annual_income': 72000},
 'credit_file': {'credit_score': 720,
  'existing_monthly_debt_payments': 400,
  'missed_payments_last_12_months': 0,
  'bankruptcy_discharged_on': None},
 'documents': [{'type': 'id',
   'name': "Driver's license",
   'content': "State driver's license for Maya Patel, valid until 2030."},
  {'type': 'payslip',
   'name': 'Last three payslips',
   'content': 'Northwind Logistics payslips for Maya Patel, July to September 2026. Annual salary $72,000, paid monthly at $6,000.00 gross.'},
  {'type': 'bank_statement',
   'name': 'Bank statem

Here is the underwriter's decision on that application, which we'll use to validate the LLM results.

In [6]:
labels[applications[0]["application_id"]]

{'application_id': 'APP-5001',
 'decision': 'approve',
 'approved_amount': 10000.0,
 'reason': 'Income limit 40% of $72,000 is $28,800. DTI of 12.2% is within 40%. Approve $10,000 (section 9).',
 'tests': 'A routine approval with every document attached',
 'difficulty': 'routine'}

### Explore the data

In [7]:
rows = []
for application in applications:
    label = labels[application["application_id"]]
    rows.append({
        "application_id": application["application_id"],
        "product": application["product"],
        "purpose": application["purpose"],
        "amount_requested": application["amount_requested"],
        "credit_score": application["credit_file"]["credit_score"],
        "difficulty": label["difficulty"],
        "decision": label["decision"],
        "approved_amount": label["approved_amount"],
        "tests": label["tests"],
    })

data = pd.DataFrame(rows)
data

,application_id,product,purpose,amount_requested,credit_score,difficulty,decision,approved_amount,tests
0,APP-5001,personal,home improvements,10000,720,routine,approve,10000.0,A routine approval with every document attached
1,APP-5002,car,car purchase,25000,760,routine,approve,25000.0,A routine car loan approval
2,APP-5003,personal,debt consolidation,6000,560,routine,decline,0.0,A credit score below 580 is a hard decline
3,APP-5004,personal,medical bills,8000,650,routine,decline,0.0,A bankruptcy discharged 10 months ago is a har...
4,APP-5005,personal,wedding,7500,735,routine,refer,0.0,Missing payslips means a referral
5,APP-5006,personal,home improvements,20000,650,hard,approve,12000.0,"The income limit at 25% cuts the loan to $12,000"
6,APP-5007,personal,debt consolidation,15000,700,hard,approve,12000.0,DTI over 40% means reducing the loan until it ...
7,APP-5008,personal,vacation,8000,690,hard,decline,0.0,Existing debts leave no room for any loan abov...
8,APP-5009,personal,furniture,12000,710,hard,refer,0.0,Stated income 21% above the payslips is a refe...
9,APP-5010,personal,home improvements,32000,745,hard,approve,31000.0,"Stated income within 10%, but the income limit..."


In [8]:
fig = px.histogram(data, x="product", color="decision", title="Applications by product")
fig.show()

In [9]:
fig = px.scatter(data, x="credit_score", y="amount_requested", color="decision", hover_data=["application_id", "tests"], title="Credit score against amount requested")
fig.show()

## Part 2: Running with some LLMs

A Pydantic class describes the reply so we can use Structured Outputs. The rationale comes first so the model works through the policy before it decides.

In [10]:
class LoanDecision(BaseModel):
    """An underwriter's decision on one loan application."""

    rationale: str = Field(description="2-3 sentences to explain the decision, with the relevant sections of the lending policy and any calculation")
    decision: Literal["approve", "decline", "refer"]
    approved_amount: float = Field(description="The amount to lend in US dollars, or 0 for a decline or a referral")

## The prompt

The system prompt has the instructions and the whole lending policy. The user prompt has the application as JSON, including the credit file and the documents.

In [11]:
INSTRUCTIONS = """You are an underwriter at BrightPath Lending.
Decide the loan application you are given, using the lending policy below.

Choose one decision:
- approve: no hard decline or referral applies and the approved amount is at least the product minimum. Work out the approved amount using section 9.
- decline: a hard decline applies, or the most the applicant can borrow is below the product minimum.
- refer: a senior underwriter needs to look at it, for one of the reasons in section 6.
"""

SYSTEM_PROMPT = f"""{INSTRUCTIONS}
{lending_policy}
"""


def make_user_prompt(application):
    """The application, as JSON."""
    return f"""# Application

{json.dumps(application, indent=2)}
"""


def make_messages(application):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(application)},
    ]

### Calling the LLM

In [12]:
def decide(application, model, effort):
    """Ask one model to decide one application."""
    start = time.time()
    response = openrouter.chat.completions.parse(
        model=model,
        messages=make_messages(application),
        response_format=LoanDecision,
        reasoning_effort=effort,
        extra_body={"provider": {"require_parameters": True}},
    )
    answer = response.choices[0].message.parsed
    return {
        "model": model,
        "effort": effort,
        "technique": "LLM",
        "application_id": application["application_id"],
        "decision": answer.decision,
        "approved_amount": answer.approved_amount,
        "rationale": answer.rationale,
        "cost": response.usage.cost,
        "seconds": round(time.time() - start, 1),
    }

In [13]:
first = decide(applications[0], "openai/gpt-6-luna", "low")
first

{'model': 'openai/gpt-6-luna',
 'effort': 'low',
 'technique': 'LLM',
 'application_id': 'APP-5001',
 'decision': 'approve',
 'approved_amount': 10000.0,
 'rationale': 'No hard decline or referral applies: the required documents are present, verified annual income is $72,000, and the credit score is 720. Under sections 7–9, the income limit is $28,800 and the requested $10,000 loan has an estimated monthly payment of $330, making DTI ($400 + $330) / $6,000 = 12.2%, below the 40% limit; approve the full requested amount.',
 'cost': 0.000290175,
 'seconds': 4.7}

### Now running multiple models

In [14]:
def is_right(answer, label):
    """True when the decision matches the underwriter's and the amount is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["approved_amount"] - label["approved_amount"]) <= 1


def run(model, effort, applications, labels):
    """Run the model on the applications and compare against the labels."""
    results = []
    for application in applications:
        answer = decide(application, model, effort)
        label = labels[application["application_id"]]
        results.append({**answer, "difficulty": label["difficulty"], "right": is_right(answer, label)})
    right = sum(result["right"] for result in results)
    cost = sum(result["cost"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    print(f"{model} at {effort}: {right} of {len(results)} right for ${cost:.4f} in {seconds:.0f} seconds")
    return results

In [15]:
luna_low = run("openai/gpt-6-luna", "low", applications, labels)

openai/gpt-6-luna at low: 39 of 41 right for $0.0063 in 149 seconds


In [16]:
luna_medium = run("openai/gpt-6-luna", "medium", applications, labels)

openai/gpt-6-luna at medium: 40 of 41 right for $0.0069 in 157 seconds


In [17]:
# Add your own experiments here!

In [18]:
gemini = run("google/gemini-3.5-flash-lite", "low", applications, labels)

google/gemini-3.5-flash-lite at low: 31 of 41 right for $0.0336 in 57 seconds


## Results by difficulty

The hard applications need the model to apply the rules in the right order or do the affordability arithmetic. The harder ones test the exact boundaries and two-step reductions.

In [19]:
scored = pd.DataFrame(luna_low + luna_medium + gemini)
scored.pivot_table(index=["model", "effort"], columns="difficulty", values="right", aggfunc="sum", sort=False)[["routine", "hard", "harder"]]

difficulty                           routine  hard  harder
model                        effort                       
openai/gpt-6-luna            low           6    19      14
                             medium        6    19      15
google/gemini-3.5-flash-lite low           6    15      10

## Where a model got it wrong

In [20]:
def show_misses(results, labels):
    """Show each application a model got wrong, with its rationale and the underwriter's reason."""
    for result in results:
        if not result["right"]:
            label = labels[result["application_id"]]
            display(Markdown(f"""**{result['application_id']}, {label['difficulty']}**: {result['decision']} ${result['approved_amount']:,.2f}, where the underwriter said {label['decision']} ${label['approved_amount']:,.2f}

- Model: {result['rationale']}
- Underwriter: {label['reason']}
- Tests: {label['tests']}
"""))


show_misses(luna_low, labels)

**APP-5030, harder**: refer $0.00, where the underwriter said approve $10,000.00

- Model: No hard decline applies, but the verified annual income from the payslips is $60,000, while the stated $66,000 is 10% above verified income. Under sections 4 and 6, an income statement more than 10% above verified income requires referral; the approved amount for a referral is $0.
- Underwriter: Income limit 40% of $60,000 is $24,000. DTI of 16.6% is within 40%. Approve $10,000 (section 9).
- Tests: Stated income exactly 10% above verified is not more than 10%, so no referral


**APP-5032, harder**: decline $0.00, where the underwriter said refer $0.00

- Model: The requested personal loan is for a kiln for the applicant’s pottery studio, making it business use on a personal loan, which is prohibited under section 2 and requires a decline. The missing second tax return would otherwise trigger a referral under sections 3 and 6, but hard declines take priority under section 10.
- Underwriter: Required documents are missing: tax (section 3).
- Tests: Self-employed with only one tax return is a missing document


### What it cost

In [21]:
scored.groupby(["model", "effort"], sort=False)[["right", "cost", "seconds"]].sum()

right      cost  seconds
model                        effort                          
openai/gpt-6-luna            low        39  0.006285    148.7
                             medium     40  0.006860    157.0
google/gemini-3.5-flash-lite low        31  0.033564     57.0

### Commercial implications

The kind of mistake matters. A wrong approval, or approving too much, lends money the applicant may not repay. A wrong decline loses a good customer. A wrong referral costs an underwriter's time. This cell splits the mistakes by type.

In [22]:
def mistake(row):
    label = labels[row["application_id"]]
    if row["right"]:
        return "right"
    if row["decision"] == "approve" and (label["decision"] != "approve" or row["approved_amount"] > label["approved_amount"]):
        return "lent too much"
    if row["decision"] == "decline" and label["decision"] != "decline":
        return "wrong decline"
    if row["decision"] == "refer" and label["decision"] != "refer":
        return "needless referral"
    return "other"


scored["outcome"] = scored.apply(mistake, axis=1)
scored.pivot_table(index=["model", "effort"], columns="outcome", values="application_id", aggfunc="count", fill_value=0, sort=False)

outcome                              right  needless referral  wrong decline  \
model                        effort                                            
openai/gpt-6-luna            low        39                  1              1   
                             medium     40                  0              1   
google/gemini-3.5-flash-lite low        31                  0              3   

outcome                              other  lent too much  
model                        effort                        
openai/gpt-6-luna            low         0              0  
                             medium      0              0  
google/gemini-3.5-flash-lite low         2              5

## Part 3: Optional extras

### Sol at high

GPT-6 Sol is Luna's bigger sibling; we run it at `high` to see how much better it does.

In [23]:
sol = run("openai/gpt-6-sol", "high", applications, labels)

openai/gpt-6-sol at high: 38 of 41 right for $0.1438 in 182 seconds


### Jev

Jev, from TypeSafe AI, returns a probability for each option and a confidence instead of text. It can't work out an approved amount, so we score it on the decision alone.

In [24]:
JEV = "typesafe/jev-1.13"
JEV_QUESTION = {
    "type": "choice",
    "instructions": "You are an underwriter at BrightPath Lending. Decide `application` using `lending_policy`.",
    "criteria": {
        "approve": "No hard decline or referral applies and the applicant can borrow at least the product minimum.",
        "decline": "A hard decline applies, or the most the applicant can borrow is below the product minimum.",
        "refer": "A senior underwriter needs to look at it: a missing document, an income mismatch, a credit score of 580 to 619, a temporary resident, under 6 months in the job, or a disagreement between the story and the documents.",
    },
}

openrouter_decisions = OpenRouter(api_key=openrouter_api_key, timeout_ms=60_000)


def decide_with_jev(application):
    """Ask Jev for the decision on one application, with its confidence."""
    state = {"lending_policy": lending_policy, "application": application}
    start = time.time()
    response = openrouter_decisions.alpha.decisions.create(model=JEV, state=state, questions={"decision": JEV_QUESTION})
    answer = response.answers["decision"]
    return {
        "application_id": application["application_id"],
        "decision": answer.choice,
        "confidence": answer.confidence,
        "right": answer.choice == labels[application["application_id"]]["decision"],
        "cost": response.usage.cost,
        "seconds": round(time.time() - start, 2),
    }

In [25]:
jev = pd.DataFrame([decide_with_jev(application) for application in applications])
luna_decisions_right = sum(result["decision"] == labels[result["application_id"]]["decision"] for result in luna_low)

print(f"Jev: {jev['right'].sum()} of {len(jev)} decisions right for ${jev['cost'].sum():.4f} in {jev['seconds'].sum():.0f} seconds")
print(f"Luna at low: {luna_decisions_right} of {len(luna_low)} decisions right")

Jev: 36 of 41 decisions right for $0.0033 in 12 seconds
Luna at low: 39 of 41 decisions right


In [26]:
jev["confident"] = jev["confidence"] >= 0.9
jev.groupby("confident")["right"].agg(["sum", "count"])

,sum,count
confident,,
False,23,28
True,13,13


## Part 4: Using LangChain create_agent

Now an agent: an LLM, with tools, in a loop, to achieve a goal.

The application the agent sees leaves out the credit file. It has 3 tools: `get_credit_file` pulls the credit file from the bureau, `calculate_dti` does the affordability arithmetic and `record_decision` saves the decision. Giving the model a calculator is a common way to fix arithmetic mistakes.

In [27]:
recorded = {}
credit_files = {application["application_id"]: application["credit_file"] for application in applications}
PAYMENT_FACTORS = {24: 0.047, 36: 0.033, 60: 0.022}


@tool
def get_credit_file(application_id: str) -> str:
    """Pull the applicant's credit file: credit score, existing monthly debt payments, missed payments in the last 12 months and any bankruptcy."""
    credit_file = credit_files.get(application_id)
    return json.dumps(credit_file, indent=2) if credit_file else f"There is no credit file for {application_id}"


@tool
def calculate_dti(loan_amount: float, term_months: int, existing_monthly_debt: float, verified_annual_income: float) -> str:
    """Work out the debt-to-income ratio for a loan amount, using the payment factors in section 8 of the lending policy."""
    if term_months not in PAYMENT_FACTORS:
        return f"No payment factor for a {term_months} month term"
    payment = loan_amount * PAYMENT_FACTORS[term_months]
    dti = (existing_monthly_debt + payment) / (verified_annual_income / 12)
    return f"Estimated new payment ${payment:,.2f} a month. DTI {dti:.1%}"


@tool
def record_decision(application_id: str, decision: Literal["approve", "decline", "refer"], approved_amount: float, rationale: str) -> str:
    """Record the decision on a loan application in the lending system.

    approved_amount is the amount to lend in US dollars, or 0 for a decline or a referral.
    rationale is 2-3 sentences to explain the decision with the relevant sections of the lending policy.
    """
    recorded[application_id] = {"decision": decision, "approved_amount": approved_amount, "rationale": rationale}
    return f"Recorded {decision} on {application_id}"

The agent makes about four model calls per application. New OpenRouter accounts are limited to 20 requests a minute, so a rate limiter keeps the agent under that. Raise `requests_per_second` if your account has a higher limit.

In [28]:
# Keeps the agent under 20 requests a minute
rate_limiter = InMemoryRateLimiter(requests_per_second=0.3, check_every_n_seconds=0.1, max_bucket_size=1)

luna = ChatOpenAI(model="openai/gpt-6-luna", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, reasoning_effort="low", timeout=120, max_retries=5, rate_limiter=rate_limiter)

AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to pull the credit file, to check affordability and to record your decision.

{lending_policy}
"""


def make_application_prompt(application):
    """The application without its credit file. The agent pulls that for itself."""
    without_credit = {key: value for key, value in application.items() if key != "credit_file"}
    return f"""# Application

{json.dumps(without_credit, indent=2)}
"""


agent = create_agent(model=luna, tools=[get_credit_file, calculate_dti, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)

First, trying out one application.

In [29]:
result = agent.invoke({"messages": [{"role": "user", "content": make_application_prompt(applications[6])}]})

for message in result["messages"][1:]:
    message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_credit_file (call_lHNqhPR7B6MNyPBYV00YqdFk)
 Call ID: call_lHNqhPR7B6MNyPBYV00YqdFk
  Args:
    application_id: APP-5007
================================= Tool Message =================================
Name: get_credit_file

{
  "credit_score": 700,
  "existing_monthly_debt_payments": 1600,
  "missed_payments_last_12_months": 0,
  "bankruptcy_discharged_on": null
}
================================== Ai Message ==================================
Tool Calls:
  calculate_dti (call_Nnb2N1vgBhM0QbkAU60St5br)
 Call ID: call_Nnb2N1vgBhM0QbkAU60St5br
  Args:
    loan_amount: 15000
    term_months: 36
    existing_monthly_debt: 1600
    verified_annual_income: 60000
================================= Tool Message =================================
Name: calculate_dti

Estimated new payment $495.00 a month. DTI 41.9%
================================== Ai Message ==================================
Too

Now the whole queue. An application with no recorded decision counts as wrong.

In [30]:
def decide_with_agent(application):
    """Run the agent on one application and check what it recorded against the underwriter's decision."""
    start = time.time()
    result = agent.invoke({"messages": [{"role": "user", "content": make_application_prompt(application)}]})
    answer = recorded.get(application["application_id"], {"decision": "nothing recorded", "approved_amount": 0.0, "rationale": ""})
    label = labels[application["application_id"]]
    cost = sum(message.response_metadata["token_usage"]["cost"] for message in result["messages"] if message.type == "ai")
    return {
        "model": "openai/gpt-6-luna",
        "effort": "low",
        "technique": "Agent",
        "application_id": application["application_id"],
        **answer,
        "difficulty": label["difficulty"],
        "right": is_right(answer, label),
        "cost": cost,
        "seconds": round(time.time() - start, 1),
    }


recorded.clear()
agent_results = [decide_with_agent(application) for application in applications]

print(f"{len(recorded)} of {len(applications)} applications have a recorded decision")

40 of 41 applications have a recorded decision


In [31]:
pd.DataFrame(luna_low + agent_results).groupby(["model", "effort", "technique"], sort=False)[["right", "cost", "seconds"]].sum()

right      cost  seconds
model             effort technique                          
openai/gpt-6-luna low    LLM           39  0.006285    148.7
                         Agent         37  0.011527    518.8

In [32]:
show_misses(agent_results, labels)

**APP-5032, harder**: decline $0.00, where the underwriter said refer $0.00

- Model: Declined under section 2 because the requested personal loan is for a kiln for the applicant’s pottery studio, making it business use. This is a prohibited purpose and is a hard decline under section 5, so the missing second tax return does not change the decision.
- Underwriter: Required documents are missing: tax (section 3).
- Tests: Self-employed with only one tax return is a missing document


**APP-5034, harder**: approve $4,000.00, where the underwriter said decline $0.00

- Model: Approve the requested $4,000 personal loan for 24 months. No hard decline applies: the 2024-11-15 bankruptcy discharge was more than 24 months before the 2026-10-01 application (section 5); the 690 credit score permits a loan up to 40% of verified income under section 7, and the DTI is 10.1%, within the 40% limit (section 8).
- Underwriter: A bankruptcy was discharged within the last 24 months (section 5).
- Tests: A bankruptcy discharged 22 months ago is a decline


**APP-5035, harder**: nothing recorded $0.00, where the underwriter said decline $0.00

- Model: 
- Underwriter: The credit score of 570 is below 580 (section 5).
- Tests: A hard decline comes before the referral for a missing payslip


**APP-5037, harder**: approve $4,500.00, where the underwriter said decline $0.00

- Model: The applicant meets the hard-decline and referral checks: verified income is $30,000, credit score is 650, required documents are present, and the credit file shows no recent missed payments or bankruptcy. Under sections 7–9, the income limit is $7,500, but existing monthly debt of $900 limits the loan to $4,545.45 at the 40% DTI cap; rounded down to the nearest $500, the approved amount is $4,500.
- Underwriter: Income limit 25% of $30,000 is $7,500. DTI of 42.6% is over 40%, so the amount is reduced to fit. The most that fits is below the $5,000 minimum (section 9).
- Tests: The most that fits is $4,500, below the $5,000 car loan minimum


## Putting it all together

In [33]:
# If you skipped Part 3, take sol out of this list
experiments = [luna_low, luna_medium, gemini, sol, agent_results]

rows = []
for results in experiments:
    right = sum(result["right"] for result in results)
    cost = sum(result["cost"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    rows.append({
        "LLM": results[0]["model"],
        "reasoning": results[0]["effort"],
        "technique": results[0]["technique"],
        "accuracy": f"{right} / {len(results)}",
        "cost": f"${cost:.4f}",
        "speed": f"{seconds / len(results):.1f} seconds an application",
    })

pd.DataFrame(rows)

,LLM,reasoning,technique,accuracy,cost,speed
0,openai/gpt-6-luna,low,LLM,39 / 41,$0.0063,3.6 seconds an application
1,openai/gpt-6-luna,medium,LLM,40 / 41,$0.0069,3.8 seconds an application
2,google/gemini-3.5-flash-lite,low,LLM,31 / 41,$0.0336,1.4 seconds an application
3,openai/gpt-6-sol,high,LLM,38 / 41,$0.1438,4.4 seconds an application
4,openai/gpt-6-luna,low,Agent,37 / 41,$0.0115,12.7 seconds an application


## Ideas to take it further

- Add cases to `CASES` in `make_data.py`, or change a rule in `lending_policy.md` and in `underwrite()`, then run `python make_data.py` again.
- Compare the LLM and the agent on the hard cases only: does the `calculate_dti` tool fix the affordability mistakes?
- Find the cheapest model that gets at least 35 of 41 right with the agent.
- Swap the domain again: the same pattern of cases plus a rulebook plus human labels works for support tickets, expense claims, supplier invoices or job applications.